# Paderborn KAt → `paderborn_features.parquet` (no Colab)

Baixa os 32 rolamentos do [KAt Bearing DataCenter](https://groups.uni-paderborn.de/kat/BearingDataCenter/)
**um por vez**, descompacta, extrai as features no mesmo esquema do KAIST e apaga o bruto.
O único produto é o parquet, salvo no **Drive a cada rolamento** — se o runtime cair, roda tudo de novo e ele pula o que já está pronto.

Disco: pico ≈ 1 GB em `/content` (o Colab dá ~100 GB). Nada fica no seu Mac.

**Como usar**
1. Runtime → CPU basta (não precisa de GPU).
2. Runtime → *Run all*. Autoriza o Drive quando pedir.
3. Passada rápida (`MEAS = 5`, 640 gravações) leva ~30–60 min. Para a completa, `MEAS = 20`.
4. No fim, baixe `paderborn_features.parquet` do Drive para `kaist_distill/data/` e rode `python run_paderborn.py` no Mac.


In [ ]:
#@title 1 · Configuração
MEAS = 5          #@param {type:"integer"}   # medidas por (rolamento, ponto de operação): 5 = rápida, 20 = completa
DRIVE_DIR = "Mestrado/paderborn"   #@param {type:"string"}   # pasta no MyDrive onde o parquet vai ficar

from google.colab import drive
drive.mount("/content/drive")

import os, pathlib
OUT = pathlib.Path("/content/drive/MyDrive") / DRIVE_DIR
OUT.mkdir(parents=True, exist_ok=True)
WORK = pathlib.Path("/content/pb_work"); WORK.mkdir(exist_ok=True)
PARQUET = OUT / "paderborn_features.parquet"
print("parquet ->", PARQUET)


In [ ]:
#@title 2 · Dependências (unrar + scipy/pandas/pyarrow)
import shutil, subprocess
if shutil.which("unrar") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y unrar", shell=True, check=False)
if shutil.which("unrar") is None:   # fallback: bsdtar lê RAR via libarchive
    subprocess.run("apt-get -qq install -y libarchive-tools", shell=True, check=False)
EXTRACT = ["unrar", "x", "-o+", "-inul"] if shutil.which("unrar") else ["bsdtar", "-xf"]
print("extrator:", EXTRACT[0])
!pip -q install pyarrow scipy pandas >/dev/null
!df -h /content | tail -1


In [ ]:
#@title 3 · `paderborn_meta.py` (cópia fiel do kaist_distill/)
%%writefile paderborn_meta.py
"""Metadados do Paderborn KAt bearing dataset (tabelas 6-8 do data descriptor).

Fonte: Lessmeier et al., "Condition Monitoring of Bearing Damage in
Electromechanical Drive Systems by Using Motor Current Signals of Electric
Motors: A Benchmark Data Set for Data-Driven Classification", PHM Europe 2016.
"""

# --- 4 pontos de operação (Tabela 6) -------------------------------------
# Cada um varia UM fator em relação ao baseline N15_M07_F10. Isso é o que
# permite atribuir a quebra de cobertura a um fator específico, em vez de a
# um "load shift" agregado como no KAIST.
OPERATING = {
    "N15_M07_F10": dict(speed_rpm=1500, torque_nm=0.7, force_n=1000),  # baseline
    "N09_M07_F10": dict(speed_rpm= 900, torque_nm=0.7, force_n=1000),  # -rotação
    "N15_M01_F10": dict(speed_rpm=1500, torque_nm=0.1, force_n=1000),  # -torque
    "N15_M07_F04": dict(speed_rpm=1500, torque_nm=0.7, force_n= 400),  # -força radial
}
BASELINE_OP = "N15_M07_F10"

# --- 32 rolamentos (Tabelas 7-9) -----------------------------------------
HEALTHY       = ["K001", "K002", "K003", "K004", "K005", "K006"]
ARTIFICIAL_OR = ["KA01", "KA03", "KA05", "KA06", "KA07", "KA08", "KA09"]
ARTIFICIAL_IR = ["KI01", "KI03", "KI05", "KI07", "KI08"]
REAL_OR       = ["KA04", "KA15", "KA16", "KA22", "KA30"]
REAL_IR       = ["KI04", "KI14", "KI16", "KI17", "KI18", "KI21"]
REAL_BOTH     = ["KB23", "KB24", "KB27"]

CLASSES = ["Healthy", "OR", "IR", "Both"]      # OR = pista externa, IR = interna
CLASS_ID = {c: i for i, c in enumerate(CLASSES)}

def _build():
    m = {}
    for b in HEALTHY:                     m[b] = ("Healthy", "healthy")
    for b in ARTIFICIAL_OR:               m[b] = ("OR",      "artificial")
    for b in ARTIFICIAL_IR:               m[b] = ("IR",      "artificial")
    for b in REAL_OR:                     m[b] = ("OR",      "real")
    for b in REAL_IR:                     m[b] = ("IR",      "real")
    for b in REAL_BOTH:                   m[b] = ("Both",    "real")
    return m

BEARINGS = _build()                       # code -> (classe, origem do dano)
ALL_CODES = sorted(BEARINGS)

# --- aquisição ------------------------------------------------------------
SR_RAW = 64000.0          # vibração e corrente, síncronas
SR_TARGET = 25600.0       # reamostrado p/ bater com o KAIST (64k * 2/5 = 25.6k)
RESAMPLE_UP, RESAMPLE_DOWN = 2, 5

# nomes dos canais dentro do struct Y do .mat -> prefixo no parquet
CHANNEL_MAP = {
    "vibration_1":           ("vib", "A"),
    "phase_current_1":       ("cur", "U"),
    "phase_current_2":       ("cur", "V"),
    "temp_2_bearing_module": ("tmp", "TB"),
}

if __name__ == "__main__":
    from collections import Counter
    assert len(ALL_CODES) == 32, len(ALL_CODES)
    print(f"{len(ALL_CODES)} rolamentos | classes: {Counter(c for c, _ in BEARINGS.values())}")
    print(f"origem: {Counter(o for _, o in BEARINGS.values())}")
    print(f"gravações totais: 32 x {len(OPERATING)} ops x 20 medidas = {32*len(OPERATING)*20}")


In [ ]:
#@title 4 · `extract_paderborn.py` (cópia fiel do kaist_distill/ — NÃO editar tfeat/ffeat)
%%writefile extract_paderborn.py
#!/usr/bin/env python3
"""Extrai features do Paderborn KAt no MESMO esquema do kaist_features.parquet.

Decisão de projeto: reamostra 64 kHz -> 25,6 kHz (polifásico 2/5, exato) e usa
WINDOW=4096 / HOP=2048, idênticos ao KAIST. Assim `cost_ops` do kaist_lib vale
para os dois datasets sem nenhuma conversão, e as duas tabelas de Pareto ficam
lado a lado no paper.

Uso:
    python extract_paderborn.py --raw ~/Downloads/paderborn --out data/
    python extract_paderborn.py --raw ... --meas-per-setting 20   # passada completa
"""
import argparse, os, re, sys, time
import numpy as np, pandas as pd
from scipy.io import loadmat
from scipy import stats, signal

from paderborn_meta import (OPERATING, BEARINGS, CLASS_ID, CHANNEL_MAP,
                            SR_RAW, SR_TARGET, RESAMPLE_UP, RESAMPLE_DOWN)

WINDOW, HOP, MAXW = 4096, 2048, 50          # iguais ao extract_kaist.py
FNAME_RE = re.compile(r"^(N\d{2}_M\d{2}_F\d{2})_(K[AIB]?\d{2,3})_(\d+)$")

# ---------------------------------------------------------------- features
# Cópia fiel de extract_kaist.py -- qualquer divergência aqui invalida a
# comparação entre datasets, então NÃO "melhore" estas duas funções.
def tfeat(x):
    r = np.sqrt(np.mean(x**2)) + 1e-12; ax = np.abs(x)
    return {'mean': np.mean(x), 'std': np.std(x), 'rms': r, 'p2p': np.ptp(x),
            'kurt': stats.kurtosis(x), 'skew': stats.skew(x),
            'crest': np.max(ax)/r, 'shape': r/(np.mean(ax)+1e-12),
            'impulse': np.max(ax)/(np.mean(ax)+1e-12)}

def ffeat(x, sr):
    n = len(x); fr = np.fft.rfftfreq(n, 1/sr)
    mag = np.abs(np.fft.rfft(x))/(n+1e-12); p = mag**2; tot = p.sum()+1e-12
    pn = p/tot
    f = {'domfreq': fr[np.argmax(mag)], 'centroid': np.sum(fr*p)/tot,
         'entropy': -np.sum(pn[pn > 0]*np.log2(pn[pn > 0]+1e-12))}
    for lo, hi in [(0, 2000), (2000, 6000), (6000, sr/2)]:
        f[f'e{int(lo)}_{int(hi)}'] = p[(fr >= lo) & (fr < hi)].sum()
    return f

# ---------------------------------------------------------------- leitura
def _as_str(v):
    v = _unwrap(v)
    a = np.asarray(v).flat
    return str(a[0]).strip() if a.base is not None or np.asarray(v).size else ""

def _as_1d(v):
    if v is None:
        return None
    v = _unwrap(v)
    a = np.asarray(v).squeeze()
    if a.dtype == object:
        return None
    return a.astype(np.float64) if a.ndim == 1 and a.size > 1 else None

def _unwrap(o):
    """Desembrulha os ndarray(dtype=object) de tamanho 1 que o loadmat cria
    em volta de cada mat_struct."""
    while isinstance(o, np.ndarray) and o.dtype == object and o.size == 1:
        o = o.flat[0]
    return o

def read_mat(path):
    """Devolve {nome_do_canal: array 1-D}. Introspecta o struct em vez de
    assumir a ordem dos campos -- o layout varia entre lotes do dataset."""
    mat = loadmat(path, squeeze_me=False, struct_as_record=False)
    root = [v for k, v in mat.items() if not k.startswith("__")]
    if not root:
        raise ValueError("nenhuma variavel no .mat")
    obj = _unwrap(root[0])
    Y = getattr(obj, "Y", None)
    if Y is None:
        raise ValueError("struct sem campo Y")
    out = {}
    for raw_entry in np.asarray(Y, dtype=object).flat:
        entry = _unwrap(raw_entry)
        name = getattr(entry, "Name", None)
        if name is None:
            continue
        name = _as_str(name)
        if not name:
            continue
        arr = None
        for field in ("Data", "Raw", "Values", "y_values"):
            cand = getattr(entry, field, None)
            if cand is not None:
                arr = _as_1d(cand)
                if arr is not None:
                    break
        if arr is None:                      # fallback: maior vetor do struct
            best = None
            for field in getattr(entry, "_fieldnames", []):
                cand = _as_1d(getattr(entry, field, None))
                if cand is not None and (best is None or cand.size > best.size):
                    best = cand
            arr = best
        if arr is not None:
            out[name] = arr
    return out

def to_target_sr(x):
    """64 kHz -> 25,6 kHz, exato (2/5). Canais lentos (temperatura) passam
    direto: são constantes na escala de uma janela de 160 ms."""
    if x.size < 100:
        return x
    return signal.resample_poly(x, RESAMPLE_UP, RESAMPLE_DOWN).astype(np.float64)

# ---------------------------------------------------------------- extração
def process_file(path, stem, meta_extra):
    op, bearing, meas = FNAME_RE.match(stem).groups()
    cls, origin = BEARINGS[bearing]
    raw = read_mat(path)

    chans = {}
    for src, (kind, ch) in CHANNEL_MAP.items():
        if src not in raw:
            continue
        x = raw[src]
        chans[(kind, ch)] = to_target_sr(x) if kind in ("vib", "cur") else x

    if not any(k == "vib" for k, _ in chans):
        raise ValueError(f"{stem}: sem canal de vibração")

    n_fast = min(v.size for (k, _), v in chans.items() if k in ("vib", "cur"))
    rows, nw = [], 0
    for s in range(0, n_fast - WINDOW, HOP):
        if nw >= MAXW:
            break
        row = {'label': CLASS_ID[cls], 'class': cls, 'load': op,
               'recording_id': stem, 'window_idx': nw,
               'bearing': bearing, 'damage_origin': origin,
               'meas_idx': int(meas), **meta_extra}
        for (kind, ch), x in chans.items():
            if kind == 'tmp':
                seg = x if x.size < 100 else x[s:s+WINDOW]
                row[f'tmp_{ch}__mean'] = float(np.mean(seg))
                row[f'tmp_{ch}__std'] = float(np.std(seg))
                continue
            seg = x[s:s+WINDOW]
            for k, v in tfeat(seg).items():
                row[f'{kind}_{ch}__{k}'] = v
            for k, v in ffeat(seg, SR_TARGET).items():
                row[f'{kind}_{ch}__{k}'] = v
        rows.append(row); nw += 1
    return rows, nw

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--raw", required=True, help="pasta com os .mat (recursivo)")
    ap.add_argument("--out", default="data", help="pasta de saída")
    ap.add_argument("--meas-per-setting", type=int, default=5,
                    help="medidas por (rolamento, op). 5 = passada rápida, 20 = tudo")
    ap.add_argument("--resume", action="store_true")
    a = ap.parse_args()

    os.makedirs(a.out, exist_ok=True)
    parquet = os.path.join(a.out, "paderborn_features.parquet")

    files = []
    for dirpath, _, names in os.walk(a.raw):
        for n in names:
            if not n.lower().endswith(".mat"):
                continue
            stem = n[:-4]
            m = FNAME_RE.match(stem)
            if not m:
                continue
            op, bearing, meas = m.groups()
            if op not in OPERATING or bearing not in BEARINGS:
                continue
            if int(meas) > a.meas_per_setting:
                continue
            files.append((os.path.join(dirpath, n), stem))
    files.sort(key=lambda t: t[1])
    if not files:
        sys.exit(f"nenhum .mat reconhecido em {a.raw} -- confira se o unrar rodou")

    rows, done = [], set()
    if a.resume and os.path.exists(parquet):
        df0 = pd.read_parquet(parquet)
        rows = df0.to_dict("records"); done = set(df0["recording_id"].unique())
        print(f"[resume] {len(done)} gravações já extraídas")

    t0 = time.time(); errs = 0
    for i, (path, stem) in enumerate(files, 1):
        if stem in done:
            continue
        op = FNAME_RE.match(stem).group(1)
        try:
            r, nw = process_file(path, stem, OPERATING[op])
        except Exception as e:
            errs += 1; print(f"ERR {stem}: {repr(e)[:90]}", flush=True); continue
        rows += r; done.add(stem)
        if i % 25 == 0 or i == len(files):
            el = time.time()-t0
            print(f"{i}/{len(files)} · {len(rows)} janelas · {el:5.0f}s "
                  f"· eta {el/max(i,1)*(len(files)-i):5.0f}s", flush=True)

    df = pd.DataFrame(rows)

    # Nem todo lote do Paderborn traz os mesmos canais. Colunas ausentes viram
    # NaN e o kaist_lib recusa NaN de propósito -- então resolvemos aqui, de
    # forma explícita e auditável, em vez de deixar o treino escolher sozinho.
    nan_cols = df.columns[df.isna().any()].tolist()
    if nan_cols:
        bad_rec = df.loc[df[nan_cols].isna().any(axis=1), "recording_id"].nunique()
        frac = bad_rec / max(df.recording_id.nunique(), 1)
        if frac < 0.05:
            print(f"[strict] {bad_rec} gravações incompletas ({frac:.1%}) -> descartadas")
            df = df.dropna(axis=0)
        else:
            print(f"[strict] {len(nan_cols)} colunas ausentes em {frac:.1%} das "
                  f"gravações -> coluna descartada: {nan_cols[:6]}")
            df = df.drop(columns=nan_cols)
        df = df.reset_index(drop=True)

    df.to_parquet(parquet)
    print(f"\nSALVO {parquet}")
    print(f"  {len(df)} janelas · {df.recording_id.nunique()} gravações · {errs} erros")
    print(f"  classes: {df['class'].value_counts().to_dict()}")
    print(f"  ops:     {df['load'].value_counts().to_dict()}")
    print(f"  origem:  {df['damage_origin'].value_counts().to_dict()}")

if __name__ == "__main__":
    main()


In [ ]:
#@title 5 · Streaming: baixa → extrai → features → apaga, com checkpoint no Drive
import subprocess, shutil, time, sys, pathlib
import pandas as pd
from paderborn_meta import ALL_CODES

BASE = "https://groups.uni-paderborn.de/kat/BearingDataCenter"

def done_bearings():
    if not PARQUET.exists():
        return set()
    return set(pd.read_parquet(PARQUET, columns=["bearing"])["bearing"].unique())

def sh(cmd, **kw):
    return subprocess.run(cmd, shell=isinstance(cmd, str), **kw)

t_all = time.time()
for i, code in enumerate(ALL_CODES, 1):
    tag = f"[{i:02d}/32] {code}"
    if code in done_bearings():
        print(f"{tag} — já no parquet, pulando"); continue

    rar = WORK / f"{code}.rar"
    mat = WORK / "mat"
    shutil.rmtree(mat, ignore_errors=True); mat.mkdir()

    t0 = time.time()
    r = sh(["curl", "-fsSL", "--retry", "5", "--retry-delay", "5", "-C", "-",
            "-o", str(rar), f"{BASE}/{code}.rar"])
    if r.returncode != 0 or not rar.exists() or rar.stat().st_size < 1e6:
        print(f"{tag} — !! download falhou (rc={r.returncode}), seguindo"); rar.unlink(missing_ok=True); continue
    print(f"{tag} — baixado {rar.stat().st_size/1e6:.0f} MB em {time.time()-t0:.0f}s", end=" · ")

    t0 = time.time()
    r = sh(EXTRACT + [str(rar)], cwd=mat, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    n_mat = len(list(mat.rglob("*.mat")))
    if r.returncode != 0 or n_mat == 0:
        print(f"!! extração falhou (rc={r.returncode}, {n_mat} .mat), seguindo"); rar.unlink(missing_ok=True); continue
    print(f"{n_mat} .mat em {time.time()-t0:.0f}s", end=" · ")

    t0 = time.time()
    r = sh([sys.executable, "extract_paderborn.py", "--raw", str(mat), "--out", str(OUT),
            "--meas-per-setting", str(MEAS), "--resume"], capture_output=True, text=True)
    tail = [l for l in r.stdout.splitlines() if l.startswith(("SALVO", "ERR", "  ", "[strict]"))]
    errs = [l for l in tail if l.startswith("ERR")]
    print(f"features em {time.time()-t0:.0f}s" + (f" · {len(errs)} ERR" if errs else ""))
    for l in errs[:3]: print("     ", l)
    if r.returncode != 0:
        print("     !! extract_paderborn falhou:", r.stderr.strip().splitlines()[-1:] )

    rar.unlink(missing_ok=True); shutil.rmtree(mat, ignore_errors=True)   # libera o disco antes do próximo

print(f"\nfim em {(time.time()-t_all)/60:.0f} min")


In [ ]:
#@title 6 · Conferência final
import pandas as pd
d = pd.read_parquet(PARQUET)
print(f"{len(d):,} janelas · {d.recording_id.nunique():,} gravações · {d.bearing.nunique()}/32 rolamentos")
print("classes:", d["class"].value_counts().to_dict())
print("ops:    ", d["load"].value_counts().to_dict())
print("origem: ", d["damage_origin"].value_counts().to_dict())
print("NaN:    ", int(d.isna().sum().sum()))
print(f"\narquivo: {PARQUET}  ({PARQUET.stat().st_size/1e6:.1f} MB)")
print("esperado na passada rápida: 32 rolamentos × 4 ops × 5 medidas = 640 gravações, 50 janelas cada = 32.000 janelas")


## Depois

```bash
# no Mac
cp ~/Google\ Drive/…/Mestrado/paderborn/paderborn_features.parquet  kaist_distill/data/
cd kaist_distill && python run_paderborn.py      # -> results_paderborn/
```

Se aparecerem muitos `ERR` num lote de rolamentos, é layout de `.mat` diferente (risco nº 1 do `README_paderborn.md`): manda a mensagem de erro que eu ajusto o `read_mat`.
